# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have a limited chances of making queries in total.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## 1. collect training data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [ ]:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [ ]:
len(sequence_wt)

656

In [ ]:
def get_mutated_sequence(mut, sequence_wt):
  wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]
  #wt is first char
  #pos is second char to second to last char
  #mt is last char

  #Each row in the dataset contains:
  #First character: Wild-type amino acid (e.g., M)
  #Number: 0-based position in the sequence (e.g., 0, 5, 10)
  #Last character: Mutated amino acid (e.g., A, C, T, V)

  sequence = deepcopy(sequence_wt)
  #sequence is wildtype (most common natural type) of the target protein

  #returns the mutated version of the wildtype
  return sequence[:pos]+mt+sequence[pos+1:]

In [ ]:
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.273000,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.285700,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.215300,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.312200,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.218000,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1435,Q647F,0.912895,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1436,N648F,0.910830,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1437,A649F,0.983052,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1438,R650F,0.840307,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [ ]:
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [ ]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a prediction model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

In [ ]:
'''hyperparameters'''

seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, df, istrain=True):

        alphabet = 'ACDEFGHIKLMNPQRSTVWY'
        map_a2i = {j:i for i,j in enumerate(alphabet)}
        map_i2a = {i:j for i,j in enumerate(alphabet)}

        self.df = df

        self.num_samples = len(self.df)
        self.seq_length = len(self.df.sequence.values[0])
        self.num_channels = 20

        # TODO: replace one-hot encodings with your own encodings
        self.encodings = np.zeros((self.num_samples, 21)).astype(np.float32)
        self.targets = np.zeros(self.num_samples).astype(np.float32)

        if istrain:
            for it, (mutation, target) in enumerate(self.df[['mutant', 'DMS_score']].values):
                index = int(mutation[1:-1])
                newVal = mutation[-1]
                self.encodings[it, 0] = index / seq_length
                self.encodings[it, map_a2i[newVal] + 1] = 1
                self.targets[it] = target
        else:
            for it, mutation in enumerate(self.df['mutant'].values):
                index = int(mutation[1:-1])
                newVal = mutation[-1]
                self.encodings[it, 0] = index / seq_length
                self.encodings[it, map_a2i[newVal] + 1] = 1


    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return torch.tensor(self.encodings[idx]), torch.tensor(self.targets[idx])

In [ ]:
train_dataset = ProteinDataset(df_train)
test_dataset = ProteinDataset(df_test, istrain=False)

# split validation set
train_dataset, val_dataset = train_test_split(train_dataset, test_size=val_ratio, random_state=seed, shuffle=True)

# TODO: revise according to your own model
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
#My own class for predictive model

class ProteinNet(nn.Module):
    def __init__(self, inputDim):
        super(ProteinNet, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(inputDim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
#Train time
def train(model, train_loader, val_loader):
  epochs = 1000
  lr = .001
  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr)

  for epoch in range(epochs):
      model.train()
      lossTotal = 0
      for X, y in train_loader:
          X = X.to(device)
          y = y.to(device)
          optimizer.zero_grad()
          predictions = model(X.float())
          loss = criterion(predictions, y.float().view(-1, 1))
          loss.backward()
          optimizer.step()
          lossTotal += loss.item()
      with torch.no_grad():
          model.eval()
          lossValTotal = 0
          valPreds = []
          valTargets = []
          for X, y in val_loader:
              X = X.to(device)
              y = y.to(device)
              predictions = model(X.float())
              loss = criterion(predictions, y.float().view(-1, 1))
              lossValTotal += loss.item()

              valPreds.extend(predictions.cpu().numpy().flatten())
              valTargets.extend(y.cpu().numpy().flatten())
          spearmanVal = spearmanr(valTargets, valPreds)[0]
      print(f"Epoch {epoch+1}/{epochs}, Loss: {lossTotal/len(train_loader)}, Val Loss: {lossValTotal/len(val_loader)}, SpearMan: {spearmanVal:.4f}")

In [ ]:
# TODO: build your own prediction model to replace linear regression
# Hint: don't forget to use the validation set: you can either integrate the validation data into the training set or use it separately for early stopping

X_train, y_train = next(iter(train_loader))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

inputDim = X_train.shape[1]
model1 = ProteinNet(inputDim).to(device)
model2 = ProteinNet(inputDim).to(device)
model3 = ProteinNet(inputDim).to(device)
model4 = ProteinNet(inputDim).to(device)
model5 = ProteinNet(inputDim).to(device)

train(model1, train_loader, val_loader)
train(model2, train_loader, val_loader)
train(model3, train_loader, val_loader)
train(model4, train_loader, val_loader)
train(model5, train_loader, val_loader)


Epoch 1/1000, Loss: 0.10917475240098105, Val Loss: 0.09333404749631882, SpearMan: 0.2773
Epoch 2/1000, Loss: 0.08800392225384712, Val Loss: 0.07910502552986146, SpearMan: 0.3719
Epoch 3/1000, Loss: 0.07693222227195899, Val Loss: 0.07045345604419709, SpearMan: 0.3682
Epoch 4/1000, Loss: 0.06887099788420731, Val Loss: 0.06429005786776543, SpearMan: 0.3850
Epoch 5/1000, Loss: 0.0636555136491855, Val Loss: 0.06046821847558022, SpearMan: 0.3772
Epoch 6/1000, Loss: 0.05994926103287273, Val Loss: 0.057335230708122256, SpearMan: 0.3791
Epoch 7/1000, Loss: 0.056536730378866196, Val Loss: 0.05485475659370422, SpearMan: 0.3668
Epoch 8/1000, Loss: 0.05439011628429095, Val Loss: 0.05195599868893623, SpearMan: 0.3615
Epoch 9/1000, Loss: 0.052796038488547005, Val Loss: 0.04989998787641525, SpearMan: 0.3569
Epoch 10/1000, Loss: 0.050392567076616816, Val Loss: 0.04723895154893398, SpearMan: 0.3556
Epoch 11/1000, Loss: 0.04912015702575445, Val Loss: 0.04637010470032692, SpearMan: 0.3572
Epoch 12/1000, L

In [ ]:
def predict(model, test_loader):
  model.eval()
  totalPredictions = []

  with torch.no_grad():
      for X, y in test_loader:
          X = X.to(device)
          predictions = model(X.flatten(start_dim=1))
          totalPredictions.extend(predictions.cpu().numpy().flatten())
  return pd.DataFrame({
          'mutant': df_test['mutant'].values,
          'predicted_score': totalPredictions
      })

In [ ]:
#This cell returns a dict that maps mutants to their 5 fitness scores

model_predicts = [predict(m, test_loader) for m in [model1, model2, model3, model4, model5]]
combined = pd.concat(model_predicts)
mutant_dict = combined.groupby('mutant')['predicted_score'].apply(list).to_dict()
df_test['DMS_score_predicted'] = df_test['mutant'].apply(lambda x: np.mean(mutant_dict[x]))
print(len(mutant_dict))
print(mutant_dict)

11324
{'A115C': [0.38132885098457336, 0.3890514671802521, 0.36724454164505005, 0.36999446153640747, 0.3950697183609009], 'A115D': [0.3471226990222931, 0.34030812978744507, 0.35933205485343933, 0.33159810304641724, 0.36719387769699097], 'A115E': [0.22529888153076172, 0.22059322893619537, 0.21167023479938507, 0.2164250761270523, 0.26750388741493225], 'A115F': [0.19993986189365387, 0.1498139649629593, 0.14690682291984558, 0.19174061715602875, 0.2128283828496933], 'A115G': [0.6535431742668152, 0.6538581848144531, 0.6454077363014221, 0.659701943397522, 0.6096422672271729], 'A115H': [0.29915618896484375, 0.2706793248653412, 0.2692262828350067, 0.2600875496864319, 0.2930600047111511], 'A115I': [0.3852554261684418, 0.3830513656139374, 0.36547088623046875, 0.38819804787635803, 0.36104172468185425], 'A115K': [0.2814895808696747, 0.2708657681941986, 0.2545625865459442, 0.2740628123283386, 0.29771888256073], 'A115L': [0.15016542375087738, 0.15046323835849762, 0.1316210776567459, 0.1183685287833213

In [ ]:
#Find the top 10 based on avg fitness score:

averages = []
for mutation in mutant_dict:
  averages.append((mutation, np.mean(mutant_dict[mutation])))

averages.sort(key=lambda x: x[1], reverse=True)
print(averages[:10])

bestTenList = []
for i in range(10):
  bestTenList.append(averages[i][0])

print(bestTenList)
with open('top10.txt', 'w') as f:
  for mutant in bestTenList:
    f.write(mutant+'\n')

[('P655R', np.float64(0.920874834060669)), ('A654R', np.float64(0.9207105994224548)), ('E653R', np.float64(0.9205482363700866)), ('T652R', np.float64(0.9203854203224182)), ('E651R', np.float64(0.9202219128608704)), ('A649R', np.float64(0.9198930025100708)), ('N648R', np.float64(0.9197275757789611)), ('Q647R', np.float64(0.9195614933967591)), ('T646R', np.float64(0.9193945407867432)), ('P645R', np.float64(0.9192264914512634))]
['P655R', 'A654R', 'E653R', 'T652R', 'E651R', 'A649R', 'N648R', 'Q647R', 'T646R', 'P645R']


In [ ]:
df_pred_model1 = predict(model1, test_loader)
df_test['DMS_score_predicted'] = df_pred_model1['predicted_score']

In [ ]:
# Find top 10 based on model1 predictions
df_pred_model1 = predict(model1, test_loader)

scores = list(zip(df_pred_model1['mutant'], df_pred_model1['predicted_score']))

scores.sort(key=lambda x: x[1], reverse=True)

print(scores[:10])

bestTenList = [mutation for mutation, _ in scores[:10]]

print(bestTenList)

with open('top10.txt', 'w') as f:
    for mutant in bestTenList:
        f.write(mutant + '\n')

[('I569F', 0.9097418785095215), ('Q570F', 0.9097098112106323), ('I568F', 0.9096982479095459), ('P571F', 0.9096742868423462), ('E567F', 0.9096545577049255), ('T572F', 0.9096387028694153), ('R566F', 0.9096109867095947), ('V573F', 0.9096030592918396), ('V574F', 0.9095674753189087), ('G565F', 0.9095672965049744)]
['I569F', 'Q570F', 'I568F', 'P571F', 'E567F', 'T572F', 'R566F', 'V573F', 'V574F', 'G565F']


In [ ]:
#This cell returns a dict that maps mutants to their 5 rankings (based on fitness score)

model_predicts = []
models = [model1, model2, model3, model4, model5]

for i, m in enumerate(models):
    df_pred = predict(m, test_loader)
    df_pred = df_pred.copy()
    df_pred["model_id"] = i
    model_predicts.append(df_pred)

combined = pd.concat(model_predicts, ignore_index=True)

combined["rank"] = combined.groupby("model_id")["predicted_score"].rank(method="average", ascending=False)

rankingDict = (combined.sort_values(["mutant", "model_id"]).groupby("mutant")["rank"].apply(list).to_dict())

print(len(rankingDict))
print(rankingDict)

11324
{'A115C': [6164.0, 5673.0, 5947.0, 6171.0, 5679.0], 'A115D': [6630.0, 6311.0, 6099.0, 6689.0, 6148.0], 'A115E': [9082.0, 8964.0, 9066.0, 9085.0, 8278.0], 'A115F': [9647.0, 10953.0, 10718.0, 9551.0, 9274.0], 'A115G': [3853.0, 3308.0, 3322.0, 3531.0, 3394.0], 'A115H': [7736.0, 7787.0, 7870.0, 8326.0, 7751.0], 'A115I': [6098.0, 5750.0, 5986.0, 5938.0, 6231.0], 'A115K': [8083.0, 7783.0, 8274.0, 8026.0, 7603.0], 'A115L': [10940.0, 10937.0, 11103.0, 11183.0, 11185.0], 'A115M': [11184.0, 10753.0, 10821.0, 11122.0, 10875.0], 'A115N': [6721.0, 6905.0, 6108.0, 6306.0, 6789.0], 'A115P': [3110.0, 2963.0, 838.0, 1819.0, 3774.0], 'A115Q': [9263.0, 8724.0, 8957.0, 9306.0, 9204.0], 'A115R': [7173.0, 7201.0, 5843.0, 6500.0, 7961.0], 'A115S': [6514.0, 6305.0, 5686.0, 6265.0, 6140.0], 'A115T': [5824.0, 4848.0, 4537.0, 5025.0, 5454.0], 'A115V': [5855.0, 6988.0, 7070.0, 8218.0, 6721.0], 'A115W': [8069.0, 8153.0, 8154.0, 7905.0, 7361.0], 'A115Y': [5639.0, 5165.0, 4989.0, 5409.0, 5407.0], 'A144C': [773

In [ ]:
df_test[['mutant', 'DMS_score_predicted']].to_csv('predictions.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Select query for next round

In [ ]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
11309,P655F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.891732
11287,A654F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.891420
11271,E653F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.891107
11265,T652F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.890793
11231,E651F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.890479
...,...,...,...
9399,Q554F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.852123
9371,L553F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.851659
9364,E552F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.851196
9332,P551F,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.850732


In [ ]:
trainSet = set(df_train['mutant'])
print(len(trainSet))
print(trainSet)

1340
{'C242K', 'L304S', 'G157V', 'S483P', 'P253K', 'Q484P', 'L308F', 'V331P', 'L88P', 'Y296A', 'I191N', 'R182E', 'R650D', 'G220Y', 'C242D', 'S311A', 'H612D', 'V314N', 'A112Y', 'G254F', 'V314K', 'I258W', 'P276E', 'G254H', 'N295E', 'N289D', 'L309P', 'C242L', 'G157I', 'I255N', 'Y118G', 'L304H', 'R334Y', 'A292G', 'K216C', 'I251N', 'G148Q', 'I224K', 'V302F', 'Y89F', 'G220Q', 'Q508P', 'L335G', 'Y531P', 'H318F', 'R182I', 'L141H', 'P347K', 'F189K', 'D290F', 'F189V', 'L308Q', 'M0N', 'D158I', 'C305E', 'I224C', 'E532P', 'R334D', 'L228G', 'M48K', 'F185L', 'Y89L', 'V314D', 'V302M', 'R182Q', 'K216Q', 'A654H', 'I224V', 'G254I', 'W632H', 'Y320N', 'L228R', 'A221T', 'G297S', 'L304R', 'A292E', 'F319E', 'I255W', 'P315W', 'C242I', 'T463P', 'L335A', 'P253Q', 'Y296I', 'G195M', 'N151I', 'Y296E', 'P276Q', 'R334E', 'I247T', 'G297A', 'R133G', 'A221M', 'P315R', 'R650H', 'L308Y', 'Q227F', 'V193C', 'H126D', 'P315K', 'R182S', 'L304G', 'C305N', 'G297N', 'P253C', 'I298N', 'A112R', 'A221C', 'M48E', 'Y196G', 'P276Y', 'Y

In [ ]:
#SAMPLE BY VARIANCE (QUERY 1)

thompsonArr = []
thompsonQueries = []

for mutation in rankingDict:
  currMutationArr = rankingDict[mutation]
  var = np.var(currMutationArr, ddof=1)
  thompsonArr.append((mutation, var))

thompsonArr.sort(key=lambda x: x[1], reverse=True)
print(thompsonArr)

currIndex = 0
while len(thompsonQueries) < 100:
  if thompsonArr[currIndex][0] not in trainSet:
    thompsonQueries.append(thompsonArr[currIndex][0])
  currIndex += 1

print(len(thompsonQueries))
print(thompsonQueries)


[('G500P', np.float64(4504707.7)), ('W499P', np.float64(4502442.3)), ('S502P', np.float64(4490730.8)), ('G498P', np.float64(4489664.799999999)), ('V497P', np.float64(4487054.8)), ('G503P', np.float64(4483651.3)), ('G504P', np.float64(4480644.5)), ('Y505P', np.float64(4479951.300000001)), ('I496P', np.float64(4473804.5)), ('V506P', np.float64(4465818.699999999)), ('F507P', np.float64(4455539.3)), ('D494P', np.float64(4450375.199999999)), ('S493P', np.float64(4443956.7)), ('S492P', np.float64(4438410.7)), ('Q508P', np.float64(4437367.3)), ('K490P', np.float64(4429713.800000001)), ('K509P', np.float64(4425582.7)), ('G489P', np.float64(4423014.8)), ('A510P', np.float64(4422589.7)), ('N488P', np.float64(4413891.7)), ('Y511P', np.float64(4411586.2)), ('I487P', np.float64(4406405.7)), ('L512P', np.float64(4399276.3)), ('E513P', np.float64(4392229.5)), ('N486P', np.float64(4380895.7)), ('F514P', np.float64(4367646.3)), ('F515P', np.float64(4353839.699999999)), ('Q484P', np.float64(4347866.2)),

In [ ]:
#SANITY CHECK (Make sure there are none from current test set)

for mutant in thompsonQueries:
  if mutant in trainSet:
    print(mutant)

In [ ]:
with open('query.txt', 'w') as f:
  for mutant in thompsonQueries:
    f.write(mutant+'\n')

In [ ]:
#SAMPLE BY UCB (QUERY 2)

ucbArr = []
ucbQueries = []

for mutation in rankingDict:
  currMutationArr = rankingDict[mutation]
  std = np.std(currMutationArr, ddof=1)
  mean = np.mean(currMutationArr)
  ucbScore = mean - (4 * std)
  ucbArr.append((mutation, ucbScore))

ucbArr.sort(key=lambda x: x[1])
print(ucbArr)

currIndex = 0
while len(ucbQueries) < 100:
  if ucbArr[currIndex][0] not in trainSet:
    ucbQueries.append(ucbArr[currIndex][0])
  currIndex += 1

print(len(ucbArr))
print(ucbQueries)

[('P655H', np.float64(-7731.72977714244)), ('A654H', np.float64(-7697.82328773932)), ('E653H', np.float64(-7657.191154320303)), ('T652H', np.float64(-7613.359320756201)), ('E651H', np.float64(-7578.296143168789)), ('R650H', np.float64(-7537.311876311471)), ('A649H', np.float64(-7497.523780255446)), ('N648H', np.float64(-7456.377902927291)), ('P655D', np.float64(-7448.242427593406)), ('A654D', np.float64(-7430.068616220439)), ('Q647H', np.float64(-7416.982024343143)), ('E653D', np.float64(-7402.694857243298)), ('T652D', np.float64(-7379.465428405594)), ('T646H', np.float64(-7378.032944433387)), ('E651D', np.float64(-7357.801861196627)), ('P645H', np.float64(-7336.653842144537)), ('R650D', np.float64(-7335.935280661588)), ('A649D', np.float64(-7312.223290452193)), ('R644H', np.float64(-7301.354369565054)), ('N648D', np.float64(-7288.0020542315015)), ('Q647D', np.float64(-7272.426565323264)), ('N643H', np.float64(-7263.11381646566)), ('T646D', np.float64(-7247.947036862015)), ('L642H', np

In [ ]:
#SANITY CHECK (Make sure there are none from current test set)

for mutant in ucbQueries:
  if mutant in trainSet:
    print(mutant)

In [ ]:
with open('query.txt', 'w') as f:
  for mutant in ucbQueries:
    f.write(mutant+'\n')

In [ ]:
#GREEDY SAMPLING (QUERY 3)

greedyArr = []
greedyQueries = []

for mutation in rankingDict:
  currMutationArr = rankingDict[mutation]
  mean = np.mean(currMutationArr)
  greedyArr.append((mutation, mean))

greedyArr.sort(key=lambda x: x[1])
print(greedyArr)

currIndex = 0
while len(greedyQueries) < 100:
  if greedyArr[currIndex][0] not in trainSet:
    greedyQueries.append(greedyArr[currIndex][0])
  currIndex += 1
print(len(greedyQueries))
print(greedyQueries)

[('P655F', np.float64(37.0)), ('A654F', np.float64(38.4)), ('E653F', np.float64(40.2)), ('T652F', np.float64(41.4)), ('E651F', np.float64(43.2)), ('R650F', np.float64(44.6)), ('A649F', np.float64(46.4)), ('N648F', np.float64(47.6)), ('Q647F', np.float64(49.2)), ('T646F', np.float64(51.0)), ('P645F', np.float64(52.6)), ('R644F', np.float64(53.8)), ('N643F', np.float64(55.6)), ('L642F', np.float64(57.2)), ('L641F', np.float64(58.8)), ('E640F', np.float64(60.0)), ('L639F', np.float64(62.0)), ('T638F', np.float64(63.4)), ('D637F', np.float64(65.0)), ('E636F', np.float64(66.4)), ('V635F', np.float64(68.0)), ('V634F', np.float64(69.6)), ('Q633F', np.float64(70.8)), ('W632F', np.float64(72.8)), ('L631F', np.float64(74.0)), ('C630F', np.float64(75.6)), ('N629F', np.float64(77.0)), ('D628F', np.float64(78.6)), ('L627F', np.float64(80.0)), ('P626F', np.float64(81.4)), ('D624F', np.float64(83.6)), ('N623F', np.float64(85.0)), ('D622F', np.float64(86.4)), ('V621F', np.float64(88.0)), ('L620F', np.

In [ ]:
#SANITY CHECK (Make sure there are none from current test set)

for mutant in greedyQueries:
  if mutant in trainSet:
    print(mutant)

In [ ]:
with open('query.txt', 'w') as f:
  for mutant in greedyQueries:
    f.write(mutant+'\n')